# MIMaaS Demonstration

## Initializing the API, Login and Upload

Make sure you have installed the MIMAAS api after cloning it. You can do so by:

```bash
cd mimaas-api
pip install -e .
```

In [1]:
from mimaas import MIMaaSClient

client = MIMaaSClient()


In [ ]:
# Registration requires a single-use invite token from an admin.
api_token = client.register(
    username="my_username",
    email="my_email@provider.com",
    first_name="Myname",
    surname="Mysurname",
    password="MySecretPassword1!",  # Pwd needs Capital letters, special characters and a number
    invite_token="my_token",
)

In [ ]:
# Or login if you already have an account
api_token = client.login("my_username", "MySecretPassword1!")

In [6]:
# Show all boards that are currently available
boards = client.list_boards()
for board in boards:
    print(f"{board.name}: {board.ram_size_kb}KB RAM")

nrf5340dk_1: 448KB RAM
nrf5340dk_3: 448KB RAM
nrf52833dk_1: 128KB RAM
nrf52840dk_1: 256KB RAM
nucleo_l476rg_1: 96KB RAM


## Overview of your requests

In [7]:
# list_requests() returns everything on your account; pass status= to filter
# to just "pending", "processing", "done", or "error".
import pandas as pd

requests_df = pd.DataFrame([
    {
        "id": r.id,
        "created_at": r.created_at.astimezone() if r.created_at else None,  # server sends UTC; show local time
        "model": r.model_filename,
        "status": r.status,
    }
    for r in client.list_requests()
])
requests_df

,id,created_at,model,status
0,16408,2026-09-03 11:22:34.727993+02:00,model_2_unsupported_resize_bilinear.tflite,error
1,16412,2026-09-03 11:22:35.238018+02:00,model_1_unsupported_prelu.tflite,error
2,16414,2026-09-03 11:22:35.414062+02:00,model_2_unsupported_resize_bilinear.tflite,error
3,16415,2026-09-03 11:22:36.252539+02:00,model_3_too_large_flash.tflite,error
4,16417,2026-09-03 11:22:36.430019+02:00,model_1_unsupported_prelu.tflite,error
...,...,...,...,...
147,16856,2026-09-04 14:04:45.094981+02:00,model_4_large_activations.tflite,error
148,16857,2026-09-04 14:07:53.158090+02:00,model_6_compatible_int8.tflite,done
149,16858,2026-09-04 15:32:59.871202+02:00,tiny_cnn_int8.tflite,error
150,16859,2026-09-04 15:33:27.917940+02:00,tiny_cnn_int8.tflite,done


## Choose an AI model

### Use pre-built models

In [3]:
# Explore some example models in the test_models folder!

model_path = "test_models/tiny_cnn_int8.tflite"

### Build your own model

To build a model in this notebook, please install: 

```bash
pip install tensorflow pydot numpy
```

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import plot_model
import numpy as np
import os

I0000 00:00:1788532060.940685 2004855 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788532060.980025 2004855 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788532061.969529 2004855 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
def dense_block(units=32, activation="relu"):
    return layers.Dense(units, activation=activation)

def conv_block(filters=16, kernel_size=3, activation="relu"):
    return layers.Conv2D(filters, kernel_size, activation=activation, padding="same")

def pooling_block(pool_size=2):
    return layers.MaxPooling2D(pool_size)

def flatten_block():
    return layers.Flatten()

def output_block(num_classes=10):
    return layers.Dense(num_classes, activation="softmax")

In [3]:
# Example model definition (user can edit this list)
blocks = [
    conv_block(8, 3),
    pooling_block(4),
    flatten_block(),
    dense_block(64),
    dense_block(32),
    dense_block(128),
    output_block(10)
]

# Build model dynamically
model = keras.Sequential([layers.Input(shape=(28, 28, 1))] + blocks)
model.summary()


E0000 00:00:1788532063.535843 2004855 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1788532063.536073 2004931 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1788532063.556874 2004855 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 7, 7, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 392)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        25,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,826 (128.23 KB)

 Trainable params: 32,826 (128.23 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
plot_model(model, show_shapes=True, show_layer_names=True, dpi=64)

#### Train the model

In [5]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=1, validation_split=0.1)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


W0000 00:00:1788532071.526831 2004855 cpu_allocator_impl.cc:82] Allocation of 169344000 exceeds 10% of free system memory.


1688/1688 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9106 - loss: 0.2807 - val_accuracy: 0.9727 - val_loss: 0.0885


#### Convert the model to a microcontroller-compatible format

In [6]:
# Taking a slice from the training data as representative data set
def representative_dataset():
    for i in range(100):
        yield [x_train[i:i+1]]

# You can choose to quantize the model ("convert" to uint8)
# or keep it to float32. However float32 might cause the model to exceed available
# storage on MCU. Go ahead and try it out! ;)
quantize = True


converter = tf.lite.TFLiteConverter.from_keras_model(model)

if quantize:
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

tflite_model = converter.convert()

selfmade_model_path = "selfmade_model.tflite"
with open(selfmade_model_path, "wb") as f:
    f.write(tflite_model)

print("TFLite model exported (quantized =", quantize, ")")

model_path = selfmade_model_path

INFO:tensorflow:Assets written to: /tmp/tmpsw520ugd/assets


INFO:tensorflow:Assets written to: /tmp/tmpsw520ugd/assets


Saved artifact at '/tmp/tmpsw520ugd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  130580842065136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842332400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842337152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842337504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842338208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842338912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842340848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842340144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842342432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130580842341728: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/ankilab/mimaas_website/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1788532078.287660 2004855 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1788532078.287673 2004855 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1788532078.287870 2004855 reader.cc:83] Reading SavedModel from: /tmp/tmpsw520ugd
I0000 00:00:1788532078.288173 2004855 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1788532078.288178 2004855 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpsw520ugd
I0000 00:00:1788532078.291527 2004855 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1788532078.292244 2004855 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1788532078.311200 2004855 loader.cc:220] Running initialization op o

TFLite model exported (quantized = True )


## Upload Model and receive results

Choose your model and the board you want to test it on!

In [4]:
request = client.submit_request(
    model_path=model_path,
    board="nrf5340dk"
)

Completion can take up to 4 minutes, depending on the model size and latency
This process includes firmware building, model conversion, flashing and power consumption measurement

In [5]:
results = client.wait_for_completion(request.id, verbose=True)

[   0.5s] Request #16863: pending
[  11.3s] Request #16863: processing
[ 216.2s] Request #16863: done


In [13]:
# Get detailed reports for ram / rom usage and power consumption
client.download_power_summary(request.id, 'power.csv')
client.download_ram_report(request.id, 'ram.json')
client.download_power_samples(request.id, 'power_samples.zip')
client.download_all_artifacts(request.id, 'all_artifacts.zip')

## Visualize Power Consumption plots

In [14]:
# The raw power samples come down as a zip -- unzip to get the CSV out.
import zipfile

with zipfile.ZipFile('power_samples.zip') as z:
    z.extract('ppk2_samples.csv')

# Visualize with the library's own power-analysis dashboard
# (requires pandas + plotly: pip install -e ".[viz]")
from mimaas.viz import plot_power_analysis

fig = plot_power_analysis('ppk2_samples.csv')
fig.show()